# IMTS Predictive Quality Analysis
## 28-Day Concrete Strength Failure Risk Classification

### Objective
Evaluate whether IMTS concrete test data can identify tests at elevated risk of failing the 28-day required/design strength.

### Analytical failure definition
**FailureFlag28 = 1** when actual 28-day strength is below the applicable required/design strength; otherwise **0**.

> This is an analytical failure proxy for modeling, not the complete engineering acceptance standard.

### Feature sets
1. **Day 0 — Field + Required Strength**
2. **Day 7 — Day 0 + 7-Day Strength**
3. **Full Context + Day 7 — Day 0 + Day 7 + Supplier / Plant / Mix history**

### Validation method
- Same valid Day-7 records are used for all three feature sets.
- 5-fold cross-validation is grouped by project.
- Supplier / Plant / Mix failure-rate encoding is rebuilt inside each training fold.
- Because failures are rare, **PR-AUC is the primary ranking metric**.

## 1. Setup and Fabric Lakehouse Path
Attach the Lakehouse containing the cleaned Field Core CSV. Update `INPUT_PATH` below if your Lakehouse path is different.

In [ ]:
# If XGBoost is not already available in your Fabric environment:
# %pip install xgboost

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
import time
from typing import Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

print("Libraries loaded successfully.")

## 2. Configuration
The initial report threshold is 0.50 only for model comparison. A later section evaluates many thresholds using out-of-fold predictions.

In [ ]:
INPUT_PATH = (
    "/lakehouse/default/Files/field_core_outputs/"
    "field_core_clean/field_core_clean_with_required.csv"
)

TARGET_STRENGTH = "AverageActualStrength28_psi"
REQUIRED_STRENGTH = "ApplicableSpecifiedStrength28"
RISK_TARGET = "FailureFlag28"

RANDOM_STATE = 42
OUTER_CV_FOLDS = 5
TARGET_ENCODING_FOLDS = 5
TARGET_ENCODING_SMOOTHING = 20.0
REPORT_THRESHOLD = 0.50

FIELD_FEATURES = [
    "EffectiveSlump_in", "EffectiveAir_percent", "EffectiveUnitWeight_lb_ft3",
    "EffectiveConcreteTemp_F", "AmbientTemp_F", "WaterAdded_gal_per_yd3",
    "BatchToSampleMinutes", "BatchToCastMinutes", "HasWaterAdded",
    "HasAnyAfterSPMeasurement",
]
DAY0_FEATURES = FIELD_FEATURES + [REQUIRED_STRENGTH]
DAY7_AVERAGE_CANDIDATES = ["AverageActualStrength7_psi", "AverageActualStrength7"]
DAY7_COUNT_CANDIDATES = ["ActualStrength7SpecimenCount", "StandardCuredStrength7SpecimenCount"]
DAY7_FEATURES = ["Day7AverageStrength_psi", "Day7SpecimenCount", "Day7MarginToRequired_psi", "Day7ToRequiredRatio"]
CONTEXT_COLUMN_CANDIDATES = {
    "Supplier": ["SupplierId", "supplierId", "SupplierName", "supplierName"],
    "Plant": ["PlantNumber", "plantNumber", "PlantNo", "plantNo"],
    "Mix": ["MixNumber", "mixNumber", "MixNo", "mixNo"],
}
GROUP_COLUMN_CANDIDATES = ["projectId", "projectNo", "ProjectId", "ProjectNo"]

## 3. Load the Cleaned Field Core Dataset

In [ ]:
def read_csv(path: str) -> pd.DataFrame:
    for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            return pd.read_csv(path, encoding=encoding, low_memory=False)
        except UnicodeDecodeError:
            continue
    raise UnicodeError(f"Could not determine CSV encoding: {path}")

df_raw = read_csv(INPUT_PATH)
project_preview = next((c for c in GROUP_COLUMN_CANDIDATES if c in df_raw.columns), None)

overview = pd.DataFrame({
    "Metric": ["Total records", "Total columns", "Projects"],
    "Value": [
        len(df_raw),
        df_raw.shape[1],
        df_raw[project_preview].nunique(dropna=True) if project_preview else np.nan,
    ],
})
display(overview)
display(df_raw.head(10))

## 4. General Data Helpers

In [ ]:
def resolve_first_column(df: pd.DataFrame, candidates: Iterable[str]) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    return None

def require_columns(df: pd.DataFrame, columns: Iterable[str]) -> None:
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")

def numeric_series(df: pd.DataFrame, column: str) -> pd.Series:
    if column not in df.columns:
        return pd.Series(np.nan, index=df.index, dtype=float)
    s = df[column]
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce")
    return pd.to_numeric(
        s.astype("string").str.replace(",", "", regex=False).str.strip(),
        errors="coerce",
    )

def numeric_frame(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    require_columns(df, columns)
    return pd.DataFrame({c: numeric_series(df, c) for c in columns}, index=df.index)

def resolve_group_column(df: pd.DataFrame) -> str:
    col = resolve_first_column(df, GROUP_COLUMN_CANDIDATES)
    if col is None:
        raise KeyError(f"Expected one of group columns: {GROUP_COLUMN_CANDIDATES}")
    return col

def make_project_groups(df: pd.DataFrame, group_column: str) -> pd.Series:
    groups = df[group_column].astype("string").str.strip()
    missing = groups.isna() | groups.eq("")
    fallback = (
        "MISSING_PROJECT_TEST_" + df["testId"].astype("string")
        if "testId" in df.columns
        else "MISSING_PROJECT_ROW_" + pd.Series(df.index, index=df.index).astype("string")
    )
    return groups.mask(missing, fallback)

## 5. Create Day-7 Features and the Failure Target
All three feature sets are evaluated on the same records with valid 28-day actual strength, required strength, and 7-day strength.

In [ ]:
def add_day7_features(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, str | None]]:
    result = df.copy()
    avg_col = resolve_first_column(result, DAY7_AVERAGE_CANDIDATES)
    count_col = resolve_first_column(result, DAY7_COUNT_CANDIDATES)
    if avg_col is None:
        raise KeyError(f"Missing 7-day strength column: {DAY7_AVERAGE_CANDIDATES}")

    result["Day7AverageStrength_psi"] = numeric_series(result, avg_col)
    result["Day7SpecimenCount"] = numeric_series(result, count_col) if count_col else np.nan
    required = numeric_series(result, REQUIRED_STRENGTH)
    result["Day7MarginToRequired_psi"] = result["Day7AverageStrength_psi"] - required
    result["Day7ToRequiredRatio"] = result["Day7AverageStrength_psi"] / required.replace(0, np.nan)
    return result, {"source_average_column": avg_col, "source_count_column": count_col}

require_columns(df_raw, [TARGET_STRENGTH, REQUIRED_STRENGTH, *FIELD_FEATURES])
df = df_raw.copy()
actual28 = numeric_series(df, TARGET_STRENGTH)
required28 = numeric_series(df, REQUIRED_STRENGTH)
eligible = actual28.gt(0) & required28.gt(0)
eligible_count = int(eligible.sum())
df = df.loc[eligible].copy()

df, day7_metadata = add_day7_features(df)
valid_day7 = numeric_series(df, "Day7AverageStrength_psi").gt(0)
df = df.loc[valid_day7].copy()

df[RISK_TARGET] = (
    numeric_series(df, TARGET_STRENGTH) < numeric_series(df, REQUIRED_STRENGTH)
).astype(int)

failure_count = int(df[RISK_TARGET].sum())
failure_rate = float(df[RISK_TARGET].mean())

prep = pd.DataFrame({
    "Metric": ["Raw records", "Valid actual + required", "Common valid Day-7 records", "Failures", "Passes", "Failure rate"],
    "Value": [len(df_raw), eligible_count, len(df), failure_count, len(df)-failure_count, f"{failure_rate:.2%}"],
})
display(prep)
print("Day-7 average source:", day7_metadata["source_average_column"])
print("Day-7 count source:", day7_metadata["source_count_column"])

## 6. Class Distribution — Why Accuracy Is Not Enough
When failures are rare, a model can appear accurate simply by predicting PASS most of the time. Therefore PR-AUC, recall, precision, false negatives, and false positives are more useful.

In [ ]:
class_distribution = pd.DataFrame({
    "Class": ["Pass", "Failure"],
    "Count": [int((df[RISK_TARGET] == 0).sum()), int((df[RISK_TARGET] == 1).sum())],
})
class_distribution["Percent"] = (class_distribution["Count"] / len(df) * 100).round(2)
display(class_distribution)

fig, ax = plt.subplots(figsize=(7,4))
bars = ax.bar(class_distribution["Class"], class_distribution["Count"])
ax.set_title("28-Day Analytical Pass / Failure Distribution")
ax.set_ylabel("Number of Tests")
for bar, value in zip(bars, class_distribution["Count"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(), f"{value:,}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

## 7. Project-Grouped Validation

In [ ]:
group_column = resolve_group_column(df)
groups = make_project_groups(df, group_column)
unique_project_groups = int(groups.nunique())
if unique_project_groups < OUTER_CV_FOLDS:
    raise ValueError("Not enough project groups for 5-fold CV.")

display(pd.DataFrame({
    "Metric": ["Grouping column", "Records used", "Unique project groups", "Outer CV folds"],
    "Value": [group_column, len(df), unique_project_groups, OUTER_CV_FOLDS],
}))

## 8. Supplier / Plant / Mix Historical Failure Context
Historical failure-rate features are generated from training data only. Validation targets never contribute to their own historical context.

In [ ]:
@dataclass(frozen=True)
class ContextSources:
    supplier: str
    plant: str
    mix: str

def normalize_category(series: pd.Series) -> pd.Series:
    return (
        series.astype("string").fillna("__MISSING__").str.strip().str.upper()
        .str.replace(r"\s+", " ", regex=True).replace("", "__MISSING__")
    )

def resolve_context_sources(df: pd.DataFrame) -> ContextSources:
    resolved = {}
    for logical, candidates in CONTEXT_COLUMN_CANDIDATES.items():
        col = resolve_first_column(df, candidates)
        if col is None:
            raise KeyError(f"Missing context field {logical}; expected {candidates}")
        resolved[logical] = col
    return ContextSources(resolved["Supplier"], resolved["Plant"], resolved["Mix"])

def build_context_categories(df: pd.DataFrame, sources: ContextSources) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["SupplierCategory"] = normalize_category(df[sources.supplier])
    out["PlantCategory"] = normalize_category(df[sources.plant])
    out["MixCategory"] = normalize_category(df[sources.mix])
    out["SupplierPlantCategory"] = out["SupplierCategory"] + "|" + out["PlantCategory"]
    out["SupplierPlantMixCategory"] = out["SupplierCategory"] + "|" + out["PlantCategory"] + "|" + out["MixCategory"]
    return out

def smoothed_target_mapping(category, target, global_mean, smoothing):
    stats = pd.DataFrame({"category":category, "target":target}).groupby("category", dropna=False)["target"].agg(["sum","count"])
    return (stats["sum"] + smoothing * global_mean) / (stats["count"] + smoothing)

def cross_fitted_failure_target_encode(train_categories, validation_categories, y_train, train_groups):
    groups_inner = train_groups.astype("string").fillna("__MISSING_GROUP__")
    n_splits = min(TARGET_ENCODING_FOLDS, int(groups_inner.nunique()))
    if n_splits < 2:
        raise ValueError("At least two groups are required for target encoding.")
    inner = GroupKFold(n_splits=n_splits)
    global_rate = float(y_train.mean())
    encoded_train = pd.DataFrame(index=train_categories.index)
    encoded_valid = pd.DataFrame(index=validation_categories.index)
    metadata=[]

    for column in train_categories.columns:
        train_values = normalize_category(train_categories[column])
        valid_values = normalize_category(validation_categories[column])
        oof = pd.Series(np.nan, index=train_categories.index, dtype=float)
        for fit_pos, enc_pos in inner.split(train_categories, y_train, groups_inner):
            fit_idx = train_categories.index[fit_pos]
            enc_idx = train_categories.index[enc_pos]
            mapping = smoothed_target_mapping(train_values.loc[fit_idx], y_train.loc[fit_idx], global_rate, TARGET_ENCODING_SMOOTHING)
            oof.loc[enc_idx] = train_values.loc[enc_idx].map(mapping).fillna(global_rate)
        full_mapping = smoothed_target_mapping(train_values, y_train, global_rate, TARGET_ENCODING_SMOOTHING)
        counts = train_values.value_counts(dropna=False)
        encoded_train[f"{column}_FailureRate"] = oof.fillna(global_rate)
        encoded_valid[f"{column}_FailureRate"] = valid_values.map(full_mapping).fillna(global_rate)
        encoded_train[f"{column}_LogCount"] = np.log1p(train_values.map(counts).fillna(0).astype(float))
        encoded_valid[f"{column}_LogCount"] = np.log1p(valid_values.map(counts).fillna(0).astype(float))
        unknown = ~valid_values.isin(full_mapping.index)
        encoded_train[f"{column}_Unknown"] = 0
        encoded_valid[f"{column}_Unknown"] = unknown.astype(int)
        metadata.append({"ContextColumn":column, "UnknownValidationPercent":float(unknown.mean()*100), "TrainUniqueCategories":int(train_values.nunique())})
    return encoded_train, encoded_valid, pd.DataFrame(metadata)

context_sources = resolve_context_sources(df)
display(pd.DataFrame({
    "Context": ["Supplier", "Plant", "Mix"],
    "Source Column": [context_sources.supplier, context_sources.plant, context_sources.mix],
}))

## 9. Candidate Classification Models and Balanced Training

In [ ]:
def build_classification_models() -> dict[str, object]:
    return {
        "DummyPrior": DummyClassifier(strategy="prior"),
        "LogisticRegression": Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
        ]),
        "RandomForest": Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("model", RandomForestClassifier(n_estimators=400, min_samples_leaf=5, max_features=0.8, random_state=RANDOM_STATE, n_jobs=-1)),
        ]),
        "HistGradientBoosting": Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("model", HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=20, l2_regularization=1.0, random_state=RANDOM_STATE)),
        ]),
        "XGBoost": XGBClassifier(
            objective="binary:logistic", n_estimators=700, learning_rate=0.04,
            max_depth=6, min_child_weight=5, subsample=0.85, colsample_bytree=0.85,
            reg_alpha=0.0, reg_lambda=1.0, random_state=RANDOM_STATE,
            n_jobs=-1, tree_method="hist", eval_metric="logloss",
        ),
    }

def fit_balanced(model, x, y):
    # Dummy is only a baseline; weighted fitting is unnecessary.
    if isinstance(model, DummyClassifier):
        model.fit(x, y)
        return model
    weights = compute_sample_weight(class_weight="balanced", y=y)
    if isinstance(model, Pipeline):
        model.fit(x, y, model__sample_weight=weights)
    else:
        model.fit(x, y, sample_weight=weights)
    return model

display(pd.DataFrame({"Model": list(build_classification_models().keys())}))

## 10. Evaluation Metrics

In [ ]:
def classification_metrics(actual, probability, threshold=REPORT_THRESHOLD):
    predicted = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=[0,1]).ravel()
    roc = roc_auc_score(actual, probability) if len(np.unique(actual)) > 1 else np.nan
    return {
        "PR_AUC": float(average_precision_score(actual, probability)),
        "ROC_AUC": float(roc) if not np.isnan(roc) else np.nan,
        "BrierScore": float(brier_score_loss(actual, probability)),
        "Recall": float(recall_score(actual, predicted, zero_division=0)),
        "Precision": float(precision_score(actual, predicted, zero_division=0)),
        "F1": float(f1_score(actual, predicted, zero_division=0)),
        "FalseNegatives": int(fn), "FalsePositives": int(fp),
        "TruePositives": int(tp), "TrueNegatives": int(tn),
    }

# 11. Run 5-Fold Project-Grouped Cross-Validation

In [ ]:
outer = GroupKFold(n_splits=OUTER_CV_FOLDS)
fold_rows=[]
prediction_rows=[]
fold_summary_rows=[]
context_metadata_frames=[]

for fold, (train_pos, valid_pos) in enumerate(outer.split(df, groups=groups), start=1):
    print(f"Starting fold {fold}/{OUTER_CV_FOLDS}...")
    train = df.iloc[train_pos].copy()
    valid = df.iloc[valid_pos].copy()
    train_groups = make_project_groups(train, group_column)
    valid_groups = make_project_groups(valid, group_column)
    overlap = set(train_groups.astype(str)).intersection(set(valid_groups.astype(str)))
    if overlap:
        raise RuntimeError(f"Fold {fold}: project leakage detected ({len(overlap)} groups).")

    y_train = train[RISK_TARGET].astype(int)
    y_valid = valid[RISK_TARGET].astype(int)
    fold_summary_rows.append({
        "Fold":fold, "TrainRows":len(train), "ValidationRows":len(valid),
        "TrainProjects":int(train_groups.nunique()), "ValidationProjects":int(valid_groups.nunique()),
        "TrainFailureRate":float(y_train.mean()), "ValidationFailureRate":float(y_valid.mean()),
        "ProjectOverlap":len(overlap),
    })

    day0_train = numeric_frame(train, DAY0_FEATURES)
    day0_valid = numeric_frame(valid, DAY0_FEATURES)
    day7_train = numeric_frame(train, DAY7_FEATURES)
    day7_valid = numeric_frame(valid, DAY7_FEATURES)
    day7_full_train = pd.concat([day0_train, day7_train], axis=1)
    day7_full_valid = pd.concat([day0_valid, day7_valid], axis=1)

    train_context = build_context_categories(train, context_sources)
    valid_context = build_context_categories(valid, context_sources)
    enc_train, enc_valid, context_meta = cross_fitted_failure_target_encode(train_context, valid_context, y_train, train_groups)
    context_meta.insert(0, "OuterFold", fold)
    context_metadata_frames.append(context_meta)
    full_train = pd.concat([day0_train, enc_train, day7_train], axis=1)
    full_valid = pd.concat([day0_valid, enc_valid, day7_valid], axis=1)

    feature_sets = {
        "Day0_FieldPlusRequired": (day0_train, day0_valid),
        "Day7_FieldPlusRequired": (day7_full_train, day7_full_valid),
        "Full_ContextPlusDay7": (full_train, full_valid),
    }

    for feature_set, (x_train, x_valid) in feature_sets.items():
        for model_name, model in build_classification_models().items():
            start = time.perf_counter()
            model = fit_balanced(model, x_train, y_train)
            prob = model.predict_proba(x_valid)[:,1]
            elapsed = time.perf_counter() - start
            m = classification_metrics(y_valid.to_numpy(), prob)
            fold_rows.append({
                "Fold":fold, "FeatureSet":feature_set, "Model":model_name,
                "ValidationRows":len(valid), "ValidationFailureRate":float(y_valid.mean()),
                "TrainingSeconds":elapsed, **m,
            })
            pred = pd.DataFrame({
                "Fold":fold, "FeatureSet":feature_set, "Model":model_name,
                "ActualFailureFlag":y_valid.to_numpy(), "FailureProbability":prob,
            })
            for identifier in ["testId", "projectId", "projectNo", "officeId", "OfficeName"]:
                if identifier in valid.columns:
                    pred[identifier] = valid[identifier].to_numpy()
            prediction_rows.append(pred)
    print(f"Completed fold {fold}/{OUTER_CV_FOLDS}.")

print("Cross-validation complete.")

## 12. Verify Project Separation

In [ ]:
fold_summary = pd.DataFrame(fold_summary_rows)
fold_summary_display = fold_summary.copy()
fold_summary_display["TrainFailureRate"] = (fold_summary_display["TrainFailureRate"]*100).round(2)
fold_summary_display["ValidationFailureRate"] = (fold_summary_display["ValidationFailureRate"]*100).round(2)
display(fold_summary_display)
assert (fold_summary["ProjectOverlap"] == 0).all()
print("PASS: No project overlap was detected in any outer fold.")

## 13. Cross-Validation Summary — Rank Primarily by PR-AUC

In [ ]:
fold_metrics = pd.DataFrame(fold_rows)
predictions = pd.concat(prediction_rows, ignore_index=True)
context_metadata_all = pd.concat(context_metadata_frames, ignore_index=True)

summary = (
    fold_metrics.groupby(["FeatureSet","Model"], as_index=False)
    .agg(
        MeanCV_PR_AUC=("PR_AUC","mean"), StdCV_PR_AUC=("PR_AUC","std"),
        MeanCV_ROC_AUC=("ROC_AUC","mean"), StdCV_ROC_AUC=("ROC_AUC","std"),
        MeanCV_Recall=("Recall","mean"), MeanCV_Precision=("Precision","mean"), MeanCV_F1=("F1","mean"),
        MeanCV_Brier=("BrierScore","mean"),
        MeanCV_FalseNegatives=("FalseNegatives","mean"), MeanCV_FalsePositives=("FalsePositives","mean"),
    )
    .sort_values(["MeanCV_PR_AUC","MeanCV_ROC_AUC"], ascending=[False,False])
    .reset_index(drop=True)
)
summary_display = summary.copy()
for c in ["MeanCV_PR_AUC","StdCV_PR_AUC","MeanCV_ROC_AUC","StdCV_ROC_AUC","MeanCV_Recall","MeanCV_Precision","MeanCV_F1","MeanCV_Brier"]:
    summary_display[c] = summary_display[c].round(3)
for c in ["MeanCV_FalseNegatives","MeanCV_FalsePositives"]:
    summary_display[c] = summary_display[c].round(1)
display(summary_display)

## 14. Best Classifier by Feature Set

In [ ]:
best = (
    summary.sort_values(["MeanCV_PR_AUC","MeanCV_ROC_AUC"], ascending=[False,False])
    .groupby("FeatureSet", as_index=False).first()
    .sort_values("MeanCV_PR_AUC", ascending=False).reset_index(drop=True)
)
best_display = best[[
    "FeatureSet","Model","MeanCV_PR_AUC","StdCV_PR_AUC","MeanCV_ROC_AUC",
    "MeanCV_Recall","MeanCV_Precision","MeanCV_F1","MeanCV_FalseNegatives","MeanCV_FalsePositives","MeanCV_Brier"
]].copy()
for c in ["MeanCV_PR_AUC","StdCV_PR_AUC","MeanCV_ROC_AUC","MeanCV_Recall","MeanCV_Precision","MeanCV_F1","MeanCV_Brier"]:
    best_display[c] = best_display[c].round(3)
for c in ["MeanCV_FalseNegatives","MeanCV_FalsePositives"]:
    best_display[c] = best_display[c].round(1)
display(best_display)

## 15. Visual Comparison — PR-AUC

In [ ]:
feature_order = ["Day0_FieldPlusRequired", "Day7_FieldPlusRequired", "Full_ContextPlusDay7"]
plot_data = best.set_index("FeatureSet").reindex(feature_order).reset_index()
labels = ["Day 0", "Day 7", "Full Context + Day 7"]
fig, ax = plt.subplots(figsize=(9,5))
bars = ax.bar(labels, plot_data["MeanCV_PR_AUC"])
ax.set_title("Best Cross-Validated PR-AUC by Information Available")
ax.set_ylabel("Mean PR-AUC")
ax.set_ylim(0, min(1.0, max(0.2, float(plot_data["MeanCV_PR_AUC"].max())*1.2)))
ax.grid(axis="y", alpha=0.25)
for bar, value in zip(bars, plot_data["MeanCV_PR_AUC"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(), f"{value:.3f}", ha="center", va="bottom")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## 16. Recall vs Precision at Threshold 0.50

In [ ]:
x = np.arange(len(plot_data)); width=0.35
fig, ax = plt.subplots(figsize=(10,5))
ax.bar(x-width/2, plot_data["MeanCV_Recall"], width, label="Recall")
ax.bar(x+width/2, plot_data["MeanCV_Precision"], width, label="Precision")
ax.set_title("Recall vs Precision at Probability Threshold 0.50")
ax.set_ylabel("Score"); ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(0,1); ax.legend(); ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=10); plt.tight_layout(); plt.show()

# 17. Out-of-Fold Threshold Analysis
A lower alert threshold usually catches more actual failures but also creates more false positives. A higher threshold usually does the opposite. Therefore 0.50 should not automatically be the production threshold.

In [ ]:
threshold_rows=[]
thresholds = np.round(np.arange(0.05, 0.951, 0.025), 3)
for (feature_set, model_name), g in predictions.groupby(["FeatureSet","Model"]):
    actual = g["ActualFailureFlag"].to_numpy()
    prob = g["FailureProbability"].to_numpy()
    for threshold in thresholds:
        m = classification_metrics(actual, prob, float(threshold))
        threshold_rows.append({"FeatureSet":feature_set, "Model":model_name, "Threshold":threshold, **m})
threshold_table = pd.DataFrame(threshold_rows)
print(f"Threshold combinations evaluated: {len(threshold_table):,}")

## 18. Threshold Trade-Off for the Best Model in Each Feature Set

In [ ]:
selected_thresholds = [0.20,0.30,0.40,0.50,0.60,0.70]
frames=[]
for _, row in best.iterrows():
    subset = threshold_table[
        (threshold_table["FeatureSet"] == row["FeatureSet"]) &
        (threshold_table["Model"] == row["Model"]) &
        (threshold_table["Threshold"].isin(selected_thresholds))
    ].copy()
    frames.append(subset)
threshold_review = pd.concat(frames, ignore_index=True)
display(threshold_review[[
    "FeatureSet","Model","Threshold","Recall","Precision","F1",
    "FalseNegatives","FalsePositives","TruePositives","TrueNegatives"
]].round({"Recall":3,"Precision":3,"F1":3}))

## 19. Example Operating Threshold — Minimum Recall Target
The example below chooses the highest-precision threshold that still reaches at least 80% recall. The 80% target is only an example and should be decided with engineering/business stakeholders.

In [ ]:
MINIMUM_RECALL_TARGET = 0.80
rows=[]
for _, row in best.iterrows():
    candidates = threshold_table[
        (threshold_table["FeatureSet"] == row["FeatureSet"]) &
        (threshold_table["Model"] == row["Model"]) &
        (threshold_table["Recall"] >= MINIMUM_RECALL_TARGET)
    ].copy()
    if candidates.empty:
        continue
    chosen = candidates.sort_values(["Precision","Threshold"], ascending=[False,False]).iloc[0]
    rows.append({
        "FeatureSet":row["FeatureSet"], "Model":row["Model"],
        "MinimumRecallTarget":MINIMUM_RECALL_TARGET, "SelectedThreshold":chosen["Threshold"],
        "Recall":chosen["Recall"], "Precision":chosen["Precision"], "F1":chosen["F1"],
        "FalseNegatives":chosen["FalseNegatives"], "FalsePositives":chosen["FalsePositives"],
    })
operating_table = pd.DataFrame(rows)
display(operating_table.round(3))

## 20. Supplier / Plant / Mix Context Coverage

In [ ]:
context_summary = context_metadata_all.groupby("ContextColumn", as_index=False).agg(
    MeanUnknownValidationPercent=("UnknownValidationPercent","mean"),
    MaxUnknownValidationPercent=("UnknownValidationPercent","max"),
    MeanTrainUniqueCategories=("TrainUniqueCategories","mean"),
)
display(context_summary.round(1))

# 21. Business Interpretation

### Day 0
Tests whether IMTS can identify elevated 28-day failure risk using information available on placement day. This is the earliest potential warning.

### Day 7
Tests how much early strength improves risk identification. It may be more accurate, but it arrives seven days later.

### Full Context + Day 7
Tests whether historical Supplier / Plant / Mix patterns add value beyond the current test measurements.

### Key business questions
- How early is the warning available?
- How many real failures are caught?
- How many false alarms are created?
- Does the model add information beyond what an experienced project manager can already infer from the 7-day result?
- What action can still be taken when the warning is generated?

## 22. Executive Summary Template
> Project-grouped cross-validation evaluates whether IMTS data can identify elevated 28-day concrete strength failure risk on projects that were not used to train the model.
>
> Because failures are relatively uncommon, the analysis emphasizes PR-AUC, recall, precision, false negatives, and false positives rather than simple accuracy.
>
> Day 0 measures the value of information available at placement. Day 7 measures the additional value of early strength results. Full Context evaluates whether historical Supplier / Plant / Mix patterns provide additional predictive value.
>
> The probability threshold is a business decision. Lower thresholds generally catch more potential failures but create more alerts. The final threshold should be selected based on the acceptable balance between missed failures and unnecessary alerts.
>
> The analytical failure label used here is a modeling proxy and is not a replacement for engineering acceptance requirements.

## 23. Optional — Save Results to the Fabric Lakehouse

In [ ]:
# OUTPUT_DIR = (
#     "/lakehouse/default/Files/field_core_outputs/"
#     "consolidated_risk_classification_cv"
# )
# import os
# os.makedirs(OUTPUT_DIR, exist_ok=True)
# fold_metrics.to_csv(f"{OUTPUT_DIR}/classification_cv_fold_metrics.csv", index=False)
# summary.to_csv(f"{OUTPUT_DIR}/classification_cv_summary.csv", index=False)
# best.to_csv(f"{OUTPUT_DIR}/best_classifier_by_feature_set_cv.csv", index=False)
# predictions.to_csv(f"{OUTPUT_DIR}/classification_oof_predictions.csv", index=False)
# threshold_table.to_csv(f"{OUTPUT_DIR}/classification_oof_threshold_table.csv", index=False)
# fold_summary.to_csv(f"{OUTPUT_DIR}/classification_cv_fold_summary.csv", index=False)
# context_metadata_all.to_csv(f"{OUTPUT_DIR}/classification_context_encoding_metadata.csv", index=False)
# print("Results saved.")